In [3]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures

num_nodes = 4
project = "ucr-ursa-major-lesani-lab"
zone = "us-central1-c"
machine_type = "e2-highcpu-8"
image_name = "tsm-sc-image"  # your custom image
subnet = "default"
gcp_username = "tejas"

# Cleanup any existing instances with same prefix
os.system(f'gcloud compute instances delete --zone={zone} --quiet '
          f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')

# Create commands list
commands = []

for i in range(num_nodes):
    cmd = f'''
    gcloud compute instances create tsm-sc-{i:03} \
        --project={project} \
        --zone={zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=961693926925-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image={image_name},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''
    commands.append(cmd.strip())


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


# Parallel instance creation
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(run_command, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

print("All instances launched.")

# Wait a bit for IPs to propagate
import time
time.sleep(30)

# Get IPs
os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
          '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')

with open('tsm_ips.txt', 'r') as f:
    iplist = [line.strip() for line in f.readlines()]

print("🎯 Instance IPs:", iplist)


ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highcpu-8         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image=tsm-sc-image,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-e

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-000  us-central1-c  e2-highcpu-8               10.128.0.29  35.222.166.109  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP      STATUS
tsm-sc-001  us-central1-c  e2-highcpu-8               10.128.0.53  136.116.142.112  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP  STATUS
tsm-sc-003  us-central1-c  e2-highcpu-8               10.128.0.27  34.42.3.252  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-002  us-central1-c  e2-highcpu-8               10.128.0.54  34.41.13.187  RUNNING
All instances launched.
🎯 Instance IPs: ['10.128.0.29', '10.128.0.53', '10.128.0.54', '10.128.0.27']


In [2]:

# Fetch all existing instance names matching tsm-sc-*
fetch_cmd = f'''
gcloud compute instances list \
    --filter="name~'tsm-sc-'" \
    --format="value(name)"
'''
instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

print("\n➡ Existing instances to delete:", instances_to_delete)

if instances_to_delete:
    # Use parallel deletion
    def delete_instance(instance_name):
        cmd = f'''
        gcloud compute instances delete {instance_name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"Deleting: {instance_name}")
        return subprocess.call(cmd, shell=True)

    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
        concurrent.futures.wait(futures)

    print("🧹 All old tsm-sc-* instances deleted.\n")
else:
    print("✔ No previous instances found to delete.\n")



➡ Existing instances to delete: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003']
Deleting: tsm-sc-000
Deleting: tsm-sc-001
Deleting: tsm-sc-002
Deleting: tsm-sc-003


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


🧹 All old tsm-sc-* instances deleted.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].
